# Notebook 3: Cyclobutene Ring Opening — Where DFT Fails Qualitatively

**CSC Spring School on Computational Chemistry 2026**  
**Multireference Quantum Chemistry**

---

## Learning objectives

By the end of this notebook you will be able to:

- Use OpenBabel to generate a 3D starting geometry from a SMILES string
- Explain why the disrotatory ring opening of cyclobutene is a good test case for multireference methods
- Identify the correct active space for a bond-breaking reaction from the orbital picture
- Compare barrier heights from B3LYP, CASSCF(4,4), and NEVPT2 and explain the differences
- Interpret NO occupation numbers along a reaction path

---

## Background

In Notebooks 1 and 2 we saw that single-reference methods fail for **bond dissociation** — the PEC diverges at large distances. But multireference character also appears at **transition states**, even when the bond is only partially broken. This is where DFT can give qualitatively wrong barrier heights for reactions that matter in real chemistry.

### The reaction: thermal ring opening of cyclobutene

Cyclobutene undergoes thermal electrocyclic ring opening to butadiene:

$$\text{cyclobutene} \xrightarrow{\Delta} \text{butadiene}$$

The **conrotatory** pathway is thermally allowed by the Woodward–Hoffmann rules and proceeds with low barrier. The **disrotatory** pathway is thermally *forbidden* — it has a substantially higher barrier and the transition state has strong diradical character.

We will study the **disrotatory** pathway, because:
- The TS is a near-perfect diradical — the σ bond breaks homolytically
- This is exactly the same physics as H₂ dissociation, but now there is a **real barrier**
- DFT severely underestimates this barrier because it cannot describe the diradical TS
- CASSCF(4,4) + NEVPT2 gives the correct answer

### The active space

Four electrons in four orbitals — CASSCF(4,4):

| At reactant (cyclobutene) | At TS | At product (butadiene) |
|--------------------------|-------|----------------------|
| σ(C–C) | diradical orbital 1 | π₁ |
| σ*(C–C) | diradical orbital 2 | π₂* |
| π(C=C) | π bonding | π₂ |
| π*(C=C) | π antibonding | π₃* |

The same four orbitals transform continuously from reactant to product. This is the natural active space.

### The scan coordinate

We scan the **C1–C4 distance** (the breaking C–C single bond) from 1.55 Å (cyclobutene equilibrium) to 2.80 Å (approaching butadiene). At each point we compute single-point energies with all methods. This is a **rigid scan** — all other coordinates are frozen at the cyclobutene geometry. It is an approximation to the true reaction path, but sufficient to show the qualitative differences between methods.

---

## Setup

In [ ]:
import os
import re
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

ORCA_HOME     = os.environ.get('ORCA_HOME', '')
ORCA          = f"{ORCA_HOME}/orca"
HARTREE_TO_KCAL = 627.509
NPROCS        = int(os.environ.get('SLURM_CPUS_PER_TASK', '4'))

print(f"ORCA:   {ORCA}")
print(f"nprocs: {NPROCS}")

---
## Part 0: Building the starting geometry with OpenBabel

Rather than typing coordinates by hand, we use OpenBabel to generate a 3D geometry directly from a SMILES string. SMILES (Simplified Molecular Input Line Entry System) is a compact text representation of molecular structure.

Cyclobutene: `C1CC=C1`  
Read as: ring of 4 carbons, one C=C double bond, the rest single bonds.

OpenBabel will:
1. Parse the SMILES
2. Add hydrogens
3. Generate approximate 3D coordinates using a distance geometry algorithm
4. Optimise with the MMFF94 force field

This gives a reasonable starting geometry. For production work you would follow up with a DFT geometry optimisation — but for our rigid scan the force-field geometry is sufficient.

In [ ]:
import openbabel.openbabel as ob

def smiles_to_xyz(smiles, filename, ff='mmff94', steps=500):
    """Convert SMILES to 3D XYZ file via OpenBabel + force field optimisation."""
    conv    = ob.OBConversion()
    conv.SetInAndOutFormats('smi', 'xyz')
    mol     = ob.OBMol()
    conv.ReadString(mol, smiles)
    builder = ob.OBBuilder()
    builder.Build(mol)
    mol.AddHydrogens()
    ff_obj  = ob.OBForceField.FindForceField(ff)
    ff_obj.Setup(mol)
    ff_obj.ConjugateGradients(steps)
    ff_obj.GetCoordinates(mol)
    conv.WriteFile(mol, filename)
    print(f"Written {filename}  ({mol.NumAtoms()} atoms)")

smiles_to_xyz('C1CC=C1', 'cyclobutene.xyz')
!cat cyclobutene.xyz

In [ ]:
import nglview
view = nglview.show_file('cyclobutene.xyz')
view.add_representation('ball+stick')
view

### Identifying the C1–C4 bond

We need to know which atom indices correspond to the breaking C–C single bond (the bond opposite the C=C double bond in the ring). Let's print the atom coordinates and identify it:

In [ ]:
def parse_xyz(filename):
    """Return list of (element, x, y, z) from an XYZ file."""
    lines = Path(filename).read_text().strip().split('\n')
    atoms = []
    for line in lines[2:]:
        parts = line.split()
        if len(parts) == 4:
            atoms.append((parts[0],
                          float(parts[1]), float(parts[2]), float(parts[3])))
    return atoms

def bond_length(atoms, i, j):
    """Return distance between atoms i and j (0-indexed)."""
    xi, yi, zi = atoms[i][1:]
    xj, yj, zj = atoms[j][1:]
    return np.sqrt((xi-xj)**2 + (yi-yj)**2 + (zi-zj)**2)

atoms = parse_xyz('cyclobutene.xyz')
print(f"{'idx':>4}  {'elem':>4}  {'x':>8}  {'y':>8}  {'z':>8}")
for i, (el, x, y, z) in enumerate(atoms):
    print(f"{i:>4}  {el:>4}  {x:>8.4f}  {y:>8.4f}  {z:>8.4f}")

print("\nC–C distances (carbon pairs only):")
carbons = [i for i, a in enumerate(atoms) if a[0] == 'C']
for i in carbons:
    for j in carbons:
        if j > i:
            d = bond_length(atoms, i, j)
            if d < 2.0:  # only bonded pairs
                print(f"  C{i}–C{j}: {d:.4f} Å")

Look at the C–C distances above. The **shortest** C–C distance is the C=C double bond (~1.33 Å). The **breaking bond** we want to scan is the single bond on the opposite side of the ring (~1.55 Å).

**Set the indices below** based on what you see in the output above. The breaking bond is the C–C single bond that is NOT adjacent to the double bond.

In [ ]:
# ── SET THESE based on the output above ──────────────────────────────────
# The two carbon atoms forming the breaking C–C single bond
IDX_C1 = 0   # adjust if needed
IDX_C4 = 1   # adjust if needed
# ─────────────────────────────────────────────────────────────────────────

d_eq = bond_length(atoms, IDX_C1, IDX_C4)
print(f"Breaking bond C{IDX_C1}–C{IDX_C4}: {d_eq:.4f} Å  (should be ~1.54–1.57 Å)")

---
## Part 1: Building the scan geometries

We stretch the C1–C4 bond from its equilibrium value (~1.55 Å) to 2.80 Å, keeping all other atom positions fixed. This is a **rigid scan** — a simplification, but sufficient to see the qualitative differences between methods.

At each step we scale the C1–C4 vector and update only those two atom positions.

In [ ]:
def stretch_bond(atoms, i, j, new_dist):
    """Return new atom list with bond i–j stretched to new_dist.
    Midpoint is kept fixed; both atoms move symmetrically."""
    atoms = [list(a) for a in atoms]  # make mutable
    xi, yi, zi = atoms[i][1:]
    xj, yj, zj = atoms[j][1:]
    mx = (xi + xj) / 2
    my = (yi + yj) / 2
    mz = (zi + zj) / 2
    dx, dy, dz = xi - mx, yi - my, zi - mz
    norm = np.sqrt(dx**2 + dy**2 + dz**2)
    scale = (new_dist / 2) / norm
    atoms[i][1] = mx + dx * scale
    atoms[i][2] = my + dy * scale
    atoms[i][3] = mz + dz * scale
    atoms[j][1] = mx - dx * scale
    atoms[j][2] = my - dy * scale
    atoms[j][3] = mz - dz * scale
    return [tuple(a) for a in atoms]

def atoms_to_xyz_block(atoms):
    """Return geometry block suitable for ORCA input (no header)."""
    lines = []
    for el, x, y, z in atoms:
        lines.append(f"{el:2s}  {x:12.6f}  {y:12.6f}  {z:12.6f}")
    return '\n'.join(lines)

# Scan from equilibrium to 2.80 Å
r_values = np.array([1.55, 1.65, 1.75, 1.85, 1.95, 2.05,
                     2.15, 2.25, 2.40, 2.55, 2.70, 2.80])

# Verify the geometries look sensible
for r in r_values[:3]:
    new_atoms = stretch_bond(atoms, IDX_C1, IDX_C4, r)
    d_check   = bond_length(new_atoms, IDX_C1, IDX_C4)
    print(f"r = {r:.2f} Å  →  actual bond = {d_check:.4f} Å")

---
## Part 2: Running the scan

We compute energies with three methods:

| Method | Notes |
|--------|-------|
| B3LYP/def2-SVP | Restricted DFT — will fail near the TS |
| CASSCF(4,4)/def2-SVP | 4 electrons in 4 orbitals (σ, σ*, π, π*) |
| NEVPT2(4,4)/def2-SVP | CASSCF + dynamic correlation via NEVPT2 |

**Active space note:** The four orbitals are the σ/σ* of the breaking C–C bond and the π/π* of the C=C bond that becomes part of the diene. These four orbitals continuously transform into the four π orbitals of butadiene. Using `ActConstraints 0` tells ORCA not to enforce active space composition — necessary for a scan where the orbital character changes along the path.

**Starting orbitals:** For CASSCF along a scan, the standard approach is to use DFT orbitals at the first point, then pass the converged `.gbw` file to the next point (`MOREAD`). This avoids convergence problems as the active space character changes.

The scan runs 12 geometries × 3 methods. It will take ~15–20 minutes.

In [ ]:
def run_orca(label, input_text):
    """Write input, run ORCA, return path to .out file."""
    Path(f"{label}.inp").write_text(input_text)
    with open(f"{label}.out", 'w') as fh:
        subprocess.run([ORCA, f"{label}.inp"],
                       stdout=fh, stderr=subprocess.STDOUT)
    return f"{label}.out"

def get_energy(out_file):
    """Return last FINAL SINGLE POINT ENERGY from ORCA output."""
    m = re.findall(r'FINAL SINGLE POINT ENERGY\s+(-?\d+\.\d+)',
                   Path(out_file).read_text())
    return float(m[-1]) if m else float('nan')

def get_no_occupations(out_file):
    """Extract CASSCF active space NO occupation numbers (sorted descending)."""
    text   = Path(out_file).read_text()
    blocks = re.findall(
        r'Natural Orbital Occupation Numbers.*?\n((?:\s*\d+\.\d+\s*\n?)+)',
        text, re.IGNORECASE)
    if not blocks:
        return []
    return sorted([float(x) for x in blocks[-1].split()], reverse=True)

In [ ]:
# ── B3LYP scan ───────────────────────────────────────────────────────────
energies_b3lyp = []

for r in r_values:
    tag      = f"cb_{r:.2f}".replace('.', 'p')
    geom     = atoms_to_xyz_block(stretch_bond(atoms, IDX_C1, IDX_C4, r))
    out = run_orca(f"{tag}_b3lyp", f"""! B3LYP def2-SVP TightSCF
%pal nprocs {NPROCS} end
* xyz 0 1
{geom}
*
""")
    e = get_energy(out)
    energies_b3lyp.append(e)
    print(f"B3LYP  r = {r:.2f} Å  E = {e:.6f} Eh", flush=True)

energies_b3lyp = np.array(energies_b3lyp)
print("\n✓ B3LYP scan complete.")

In [ ]:
# ── CASSCF(4,4) + NEVPT2 scan ────────────────────────────────────────────
# Strategy:
#   Point 0: start from B3LYP orbitals (already in .gbw)
#   Point n>0: start from converged CASSCF orbitals of point n-1 (MOREAD)

energies_casscf = []
energies_nevpt2 = []
no_occ_all      = []   # NO occupations at each geometry

prev_gbw = None   # path to previous .gbw for MOREAD

for idx, r in enumerate(r_values):
    tag  = f"cb_{r:.2f}".replace('.', 'p')
    geom = atoms_to_xyz_block(stretch_bond(atoms, IDX_C1, IDX_C4, r))

    # Build MOREAD block if we have a previous GBW
    if prev_gbw is not None:
        moread_block = f"""%moinp "{prev_gbw}"
"""
        moread_kw = "MOREAD"
    else:
        # First point: use B3LYP orbitals from the corresponding scan point
        b3lyp_gbw  = f"{tag}_b3lyp.gbw"
        moread_block = f"""%moinp "{b3lyp_gbw}"
"""
        moread_kw = "MOREAD"

    out = run_orca(f"{tag}_casscf", f"""! RHF def2-SVP def2-SVP/C TightSCF NEVPT2 {moread_kw}
%pal nprocs {NPROCS} end
{moread_block}
%casscf
  nel          4
  norb         4
  nroots       1
  ActConstraints 0
  printlevel   3
end
* xyz 0 1
{geom}
*
""")

    # Extract energies — CASSCF energy comes before NEVPT2 in the output
    text = Path(out).read_text()

    # CASSCF total energy
    m_cas = re.findall(r'CASSCF TOTAL ENERGY:\s+(-?\d+\.\d+)', text)
    e_cas = float(m_cas[-1]) if m_cas else float('nan')

    # NEVPT2 total energy
    m_nev = re.findall(r'NEVPT2 Total Energy:\s+(-?\d+\.\d+)', text)
    e_nev = float(m_nev[-1]) if m_nev else float('nan')

    energies_casscf.append(e_cas)
    energies_nevpt2.append(e_nev)
    no_occ_all.append(get_no_occupations(out))

    # Update GBW path for next iteration
    prev_gbw = f"{tag}_casscf.gbw"

    print(f"CASSCF r = {r:.2f} Å  "
          f"E(CAS) = {e_cas:.6f}  E(NEV) = {e_nev:.6f}", flush=True)

energies_casscf = np.array(energies_casscf)
energies_nevpt2 = np.array(energies_nevpt2)
print("\n✓ CASSCF + NEVPT2 scan complete.")

---
## Part 3: Plotting the reaction path

We plot all three methods on the same graph, with energies relative to the reactant (cyclobutene at r = 1.55 Å), converted to kcal/mol.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

methods = {
    'B3LYP':         (energies_b3lyp,   '#8e44ad', '-',  'o'),
    'CASSCF(4,4)':   (energies_casscf,  '#e74c3c', '--', 's'),
    'NEVPT2(4,4)':   (energies_nevpt2,  '#27ae60', '-',  'D'),
}

for label, (E, color, ls, marker) in methods.items():
    E_rel = (E - E[0]) * HARTREE_TO_KCAL
    ax.plot(r_values, E_rel,
            color=color, ls=ls, marker=marker,
            ms=6, lw=2, label=label)

ax.axhline(0, color='grey', lw=0.5, ls=':')
ax.set_xlabel('C1–C4 distance (Å)', fontsize=13)
ax.set_ylabel('Relative energy (kcal/mol)', fontsize=13)
ax.set_title('Disrotatory ring opening of cyclobutene — def2-SVP', fontsize=13)
ax.legend(fontsize=12)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
plt.tight_layout()
plt.savefig('cyclobutene_pec.pdf')
plt.show()

### ✏️ Fill in the barrier heights

Read the maximum energy from each curve and fill in the table:

| Method | Barrier height (kcal/mol) | TS location (Å) |
|--------|--------------------------|----------------|
| B3LYP | _____ | _____ |
| CASSCF(4,4) | _____ | _____ |
| NEVPT2(4,4) | _____ | _____ |
| Experimental (disrotatory) | ~40 kcal/mol | — |

Or extract them numerically:

In [ ]:
for label, E in [('B3LYP',       energies_b3lyp),
                  ('CASSCF(4,4)', energies_casscf),
                  ('NEVPT2(4,4)', energies_nevpt2)]:
    E_rel   = (E - E[0]) * HARTREE_TO_KCAL
    barrier = E_rel.max()
    idx_ts  = E_rel.argmax()
    print(f"{label:15s}  barrier = {barrier:6.1f} kcal/mol  "
          f"at r = {r_values[idx_ts]:.2f} Å")

### ✏️ Questions — Part 3

**Q1.** By how many kcal/mol does B3LYP underestimate the NEVPT2 barrier? Express this as a percentage of the NEVPT2 barrier. Is this a chemically significant error?

*Your answer:*
```

```

**Q2.** CASSCF(4,4) overestimates the barrier relative to NEVPT2. Why? What is missing from CASSCF?

*Your answer:*
```

```

**Q3.** The scan is a rigid scan — all coordinates except C1–C4 are frozen. How would you expect a fully relaxed scan (or IRC) to change the barrier height? Would it be higher or lower, and why?

*Your answer:*
```

```

---
## Part 4: Natural orbital occupations along the reaction path

Just as in Notebook 1, we can use the CASSCF natural orbital occupation numbers to quantify the multireference character at each point along the path. The four active orbitals are σ, σ*, π, π* — at the TS we expect the σ and σ* occupations to approach 1.0.

In [ ]:
# no_occ_all[i] = [n1, n2, n3, n4] sorted descending at geometry i
no_array = np.array([occs[:4] if len(occs) >= 4
                     else [float('nan')]*4
                     for occs in no_occ_all])

fig, ax = plt.subplots(figsize=(8, 4))

colors = ['#2980b9', '#27ae60', '#e67e22', '#c0392b']
labels = ['NO 1 (σ / π₁)', 'NO 2 (π / π₂)',
          'NO 3 (π* / π₃*)', 'NO 4 (σ* / π₄*)']

for i in range(4):
    ax.plot(r_values, no_array[:, i],
            'o-', color=colors[i], lw=2, label=labels[i])

ax.axhline(1.0, color='grey', lw=0.8, ls=':', label='occ = 1.0')
ax.set_xlabel('C1–C4 distance (Å)', fontsize=13)
ax.set_ylabel('NO occupation number', fontsize=13)
ax.set_title('Active space NO occupations — CASSCF(4,4)/def2-SVP', fontsize=13)
ax.set_ylim(-0.05, 2.05)
ax.legend(fontsize=10, loc='center right')
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
plt.tight_layout()
plt.savefig('cyclobutene_no_occ.pdf')
plt.show()

### ✏️ Questions — Part 4

**Q4.** At the TS geometry (the r value with the highest energy), what are the occupation numbers of the four active orbitals? What does this tell you about the character of the TS?

*Your answer:*
```

```

**Q5.** At the reactant geometry (r = 1.55 Å), the σ occupation should be close to 2.0 and σ* close to 0.0. Is this what you see? What does this mean for the multireference character of the ground state of cyclobutene?

*Your answer:*
```

```

**Q6.** Compare the r value at which the NO occupations cross 1.0 with the r value of the barrier maximum. Are they the same? What does this tell you about the relationship between multireference character and the barrier?

*Your answer:*
```

```

**Q7.** B3LYP uses a single determinant with doubly-occupied orbitals. Given what you see in the NO occupation plot, explain *why* B3LYP must fail near the TS — not just that it does, but the fundamental reason.

*Your answer:*
```

```

---
## Part 5: When should you worry?

The cyclobutene example shows a general principle: **any reaction where a σ bond breaks homolytically will have a TS with diradical character**, and DFT will underestimate the barrier.

A practical diagnostic: run a CASSCF(2,2) or CASSCF(4,4) calculation at the suspected TS geometry and check the NO occupations. If the two frontier orbitals have occupations significantly different from (2.0, 0.0) — say, both between 1.2 and 0.8 — multireference treatment is essential.

**Reactions where this matters in practice:**

- Electrocyclic reactions (especially thermally forbidden pathways)
- Retro-Diels-Alder reactions
- Radical recombination and β-scission
- Transition metal catalysis with open-shell intermediates
- Photochemical reactions (conical intersections)

### ✏️ Final question

**Q8.** Think about a reaction relevant to your own research or field. Could it involve a TS with diradical character? What evidence would you look for before deciding whether to use a multireference method?

*Your answer:*
```

```

---
## Summary

Fill in the blanks:

1. The active space for cyclobutene ring opening is CASSCF(**___**,**___**) — the σ/σ* of the breaking bond and the π/π* of the forming diene.

2. At the TS, the two frontier NO occupations are approximately **_____** and **_____** — indicating **strong / weak** (circle one) multireference character.

3. B3LYP **underestimates / overestimates** (circle one) the barrier by approximately **_____** kcal/mol relative to NEVPT2.

4. CASSCF **underestimates / overestimates** (circle one) the barrier relative to NEVPT2 because it lacks **_____ correlation**.

5. The correct level of theory for this problem is **_____**, which captures both static and dynamic correlation.

6. A practical diagnostic for multireference character at a TS is to inspect the **_____** from a CASSCF calculation and check whether frontier orbital occupations deviate significantly from **_____** and **_____**.

---

**End of Multireference Quantum Chemistry exercises.**

For further reading on applying these methods in practice, see the ORCA manual sections on CASSCF and NEVPT2, and the review by Roos et al. on multiconfigurational quantum chemistry.